# Solution PromoPro — étape 2B : entraîner notre propre modèle sur ResPlan

Objectif : un modèle de segmentation (U-Net, encodeur ResNet-34) à 4 classes — fond, **mur**, **porte**, **fenêtre** — entraîné sur **ResPlan** (17 000 plans résidentiels vectorisés, licence **CC BY 4.0** : usage commercial permis avec attribution), exporté en **ONNX** pour tourner ensuite dans PromoPro sans Python (onnxruntime-node). Les pièces se déduisent des murs par composantes connexes ; les noms de pièces viendront d'une lecture OCR séparée.

## Avant de commencer (Google Colab gratuit)
1. Ouvrez ce carnet dans Colab : *Fichier → Importer le notebook*.
2. *Exécution → Modifier le type d'exécution → Accélérateur matériel : GPU (T4)*.
3. Exécutez les cellules dans l'ordre. La cellule 2 demande l'accès à votre Google Drive : c'est là que les **points de reprise** sont écrits à chaque époque. Quand Colab coupe la session, relancez simplement toutes les cellules : l'entraînement **reprend** au dernier point.

## Limites réelles du niveau gratuit
- Sessions coupées après ~90 min d'inactivité et au bout de quelques heures d'usage GPU par jour (variable) ; le GPU peut être indisponible aux heures de pointe.
- Avec ~13 000 images en 512 px sur T4, comptez **25 à 40 min par époque** : 15 époques ≈ 8 à 10 h de GPU, donc **3 à 6 sessions sur plusieurs jours**. Le carnet est écrit pour cela (reprise automatique).
- Pour un premier essai rapide, réduisez `NB_IMAGES` (cellule 4) à 2 000 : une époque prend alors ~5 min.

⚠️ Ce carnet a été écrit sans pouvoir être exécuté (pas de Python sur la machine de développement). La cellule 3 **inspecte** le format réel de ResPlan et la cellule 4 contient l'adaptation des clés : lisez la sortie de la cellule 3 avant de lancer la 4.

In [ ]:
# 1. Dépendances
%pip -q install segmentation-models-pytorch shapely opencv-python-headless onnx onnxruntime
import torch; print('GPU :', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
# 2. Google Drive (points de reprise) et jeu de données ResPlan (CC BY 4.0, code MIT)
import os
DOSSIER = '/content/promopro-ia'  # repli local si Drive refuse : les points de reprise sont alors perdus à la fin de la session
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    DOSSIER = '/content/drive/MyDrive/promopro-ia'
    print('Drive monté : les points de reprise iront dans', DOSSIER)
except Exception as e:
    print('Drive non monté (', e, ').')
    print('→ Essai réduit possible sans Drive, mais SANS reprise après coupure : pour l\x27entraînement complet, remontez Drive')
    print('  (icône dossier à gauche → « Monter Drive »), puis relancez cette cellule.')
os.makedirs(DOSSIER, exist_ok=True)
!git clone -q https://github.com/m-agour/ResPlan.git /content/ResPlan || true
!ls -la /content/ResPlan | head -20
# Attribution obligatoire (CC BY 4.0) : Abouagour & Garyfallidis, « ResPlan: A Large-Scale Vector-Graph Dataset of 17,000 Residential Floor Plans », 2025.

In [ ]:
# 3. Inspection du format (ne rien supposer : afficher un enregistrement)
import sys, pickle, zipfile, glob, json
sys.path.insert(0, '/content/ResPlan')
zips = glob.glob('/content/ResPlan/**/ResPlan*.zip', recursive=True)
print('archives :', zips)
if zips:
    with zipfile.ZipFile(zips[0]) as z:
        print(z.namelist()[:10]); z.extractall('/content/resplan_data')
pkls = glob.glob('/content/resplan_data/**/*.pkl', recursive=True) + glob.glob('/content/ResPlan/**/*.pkl', recursive=True)
print('pickles :', pkls[:5])
donnees = pickle.load(open(pkls[0], 'rb'))
print(type(donnees), len(donnees))
premier = donnees[0] if isinstance(donnees, list) else next(iter(donnees.values()))
print('clés du premier plan :', list(premier.keys()) if isinstance(premier, dict) else type(premier))
for k, v in (premier.items() if isinstance(premier, dict) else []):
    print(f'  {k}: {type(v).__name__} {str(v)[:160]}')
try:
    import resplan_utils; print('resplan_utils :', [n for n in dir(resplan_utils) if not n.startswith('_')])
except Exception as e:
    print('resplan_utils non importable :', e)

In [ ]:
# 4. Rastérisation : image « dessin de plan » + masque (0 fond, 1 mur, 2 porte, 3 fenêtre)
# ADAPTATION : d'après la cellule 3, renseignez comment lire, pour un plan, les polygones des murs, portes, fenêtres et pièces.
import numpy as np, cv2, random
from shapely.geometry import shape, Polygon, MultiPolygon
TAILLE = 512
NB_IMAGES = 13000   # réduire à 2000 pour un premier essai
SORTIE = '/content/resplan_raster'; os.makedirs(SORTIE, exist_ok=True)

def polygones(plan, cle):
    """Liste de polygones (listes de points) pour la classe `cle` d'un plan ResPlan.
    Hypothèse par défaut : plan[cle] est une liste de polygones (coordonnées en pixels ou en mètres),
    ou une géométrie shapely / GeoJSON. Adaptez si la cellule 3 montre autre chose."""
    v = plan.get(cle) if isinstance(plan, dict) else None
    if v is None: return []
    out = []
    for g in (v if isinstance(v, list) else [v]):
        if isinstance(g, dict) and 'type' in g: g = shape(g)
        if isinstance(g, (Polygon, MultiPolygon)):
            for p in (g.geoms if isinstance(g, MultiPolygon) else [g]): out.append(np.array(p.exterior.coords))
        elif isinstance(g, dict) and 'polygon' in g: out.append(np.array(g['polygon']))
        else:
            try: out.append(np.array(g, dtype=float).reshape(-1, 2))
            except Exception: pass
    return out

CLES = {'murs': ['walls', 'wall'], 'portes': ['doors', 'door'], 'fenetres': ['windows', 'window'], 'pieces': ['rooms', 'room', 'spaces']}
def premiere_cle(plan, candidats):
    return next((c for c in candidats if isinstance(plan, dict) and c in plan), None)

def rasteriser(plan, aleatoire):
    tout = [p for cle in CLES.values() for k in [premiere_cle(plan, cle)] if k for p in polygones(plan, k)]
    if not tout: return None
    pts = np.concatenate(tout); mn, mx = pts.min(0), pts.max(0)
    echelle = (TAILLE - 40) / max(mx - mn); decal = 20 + ((TAILLE - 40) - (mx - mn) * echelle) / 2
    def px(p): return ((p - mn) * echelle + decal).astype(np.int32)
    image = np.full((TAILLE, TAILLE, 3), 255, np.uint8); masque = np.zeros((TAILLE, TAILLE), np.uint8)
    k = premiere_cle(plan, CLES['pieces'])
    for p in (polygones(plan, k) if k else []): cv2.fillPoly(image, [px(p)], (247, 245, 240))
    for nom, classe, couleur in [('murs', 1, (25, 25, 25)), ('fenetres', 3, (120, 160, 220)), ('portes', 2, (200, 140, 60))]:
        k = premiere_cle(plan, CLES[nom])
        for p in (polygones(plan, k) if k else []):
            cv2.fillPoly(masque, [px(p)], classe); cv2.fillPoly(image, [px(p)], couleur)
    # Augmentations « vrai plan » : texte parasite, cotes, rotation légère, flou, bruit JPEG
    if aleatoire:
        for _ in range(random.randint(2, 8)):
            cv2.putText(image, random.choice(['Salon', 'Cuisine', '3,45', 'Ch. 1', 'SDB', '12 m²', 'WC']), (random.randint(30, TAILLE - 120), random.randint(30, TAILLE - 30)), cv2.FONT_HERSHEY_SIMPLEX, random.uniform(0.35, 0.7), (40, 40, 40), 1, cv2.LINE_AA)
        angle = random.uniform(-3, 3); M = cv2.getRotationMatrix2D((TAILLE / 2, TAILLE / 2), angle, 1)
        image = cv2.warpAffine(image, M, (TAILLE, TAILLE), borderValue=(255, 255, 255)); masque = cv2.warpAffine(masque, M, (TAILLE, TAILLE), flags=cv2.INTER_NEAREST, borderValue=0)
        if random.random() < 0.5: image = cv2.GaussianBlur(image, (3, 3), 0)
        if random.random() < 0.5:
            ok, enc = cv2.imencode('.jpg', image, [cv2.IMWRITE_JPEG_QUALITY, random.randint(35, 80)]); image = cv2.imdecode(enc, 1)
    return image, masque

plans = donnees if isinstance(donnees, list) else list(donnees.values())
random.seed(0); random.shuffle(plans)
n = 0
for i, plan in enumerate(plans[:NB_IMAGES]):
    r = rasteriser(plan, aleatoire=True)
    if r is None: continue
    cv2.imwrite(f'{SORTIE}/{i:05d}.png', r[0]); cv2.imwrite(f'{SORTIE}/{i:05d}_m.png', r[1]); n += 1
print('images rastérisées :', n)
assert n > 100, 'Aucune géométrie lue : adaptez CLES / polygones() d\'après la cellule 3.'

In [ ]:
# 5. Entraînement avec reprise automatique depuis Google Drive
import glob, torch, torch.nn as nn, segmentation_models_pytorch as smp
from torch.utils.data import Dataset, DataLoader
EPOQUES = 15; LOT = 8; LR = 3e-4
fichiers = sorted(glob.glob(f'{SORTIE}/*[0-9].png')); coupe = int(len(fichiers) * 0.9)
moy = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1); ecart = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
class Jeu(Dataset):
    def __init__(s, f): s.f = f
    def __len__(s): return len(s.f)
    def __getitem__(s, i):
        img = cv2.cvtColor(cv2.imread(s.f[i]), cv2.COLOR_BGR2RGB); m = cv2.imread(s.f[i].replace('.png', '_m.png'), 0)
        x = (torch.from_numpy(img).permute(2, 0, 1).float() / 255 - moy) / ecart
        return x, torch.from_numpy(m).long()
dl_train = DataLoader(Jeu(fichiers[:coupe]), batch_size=LOT, shuffle=True, num_workers=2, drop_last=True)
dl_val = DataLoader(Jeu(fichiers[coupe:]), batch_size=LOT, num_workers=2)
app = 'cuda' if torch.cuda.is_available() else 'cpu'
modele = smp.Unet('resnet34', encoder_weights='imagenet', classes=4).to(app)
opt = torch.optim.AdamW(modele.parameters(), lr=LR); scaler = torch.cuda.amp.GradScaler(enabled=app == 'cuda')
poids_classes = torch.tensor([0.3, 1.0, 3.0, 3.0]).to(app)  # portes et fenêtres sont rares : on les pèse davantage
perte = nn.CrossEntropyLoss(weight=poids_classes)
REPRISE = f'{DOSSIER}/reprise.pt'; MEILLEUR = f'{DOSSIER}/meilleur.pt'
debut, meilleur_iou = 0, 0.0
if os.path.exists(REPRISE):
    ck = torch.load(REPRISE, map_location=app); modele.load_state_dict(ck['modele']); opt.load_state_dict(ck['opt']); debut = ck['epoque'] + 1; meilleur_iou = ck['meilleur_iou']
    print(f'reprise à l\'époque {debut}, meilleur IoU {meilleur_iou:.3f}')

def iou_par_classe(pred, cible):
    return [((pred == c) & (cible == c)).sum().item() / max(1, ((pred == c) | (cible == c)).sum().item()) for c in range(4)]

for ep in range(debut, EPOQUES):
    modele.train(); total = 0
    for x, y in dl_train:
        x, y = x.to(app), y.to(app); opt.zero_grad()
        with torch.autocast(device_type=app, enabled=app == 'cuda'):
            l = perte(modele(x), y)
        scaler.scale(l).backward(); scaler.step(opt); scaler.update(); total += l.item()
    modele.eval(); ious = np.zeros(4); nb = 0
    with torch.no_grad():
        for x, y in dl_val:
            p = modele(x.to(app)).argmax(1).cpu(); ious += np.array(iou_par_classe(p, y)); nb += 1
    ious /= max(1, nb); miou = ious[1:].mean()
    print(f'époque {ep + 1}/{EPOQUES} — perte {total / len(dl_train):.3f} — IoU fond/mur/porte/fenêtre {np.round(ious, 3)} — mIoU (mur, porte, fenêtre) {miou:.3f}')
    torch.save({'modele': modele.state_dict(), 'opt': opt.state_dict(), 'epoque': ep, 'meilleur_iou': max(meilleur_iou, miou)}, REPRISE)
    if miou > meilleur_iou:
        meilleur_iou = miou; torch.save(modele.state_dict(), MEILLEUR); print('  → meilleur modèle enregistré sur Drive')

In [ ]:
# 6. Export ONNX (pour onnxruntime-node dans PromoPro, sans Python) et test sur le plan de villa
modele.load_state_dict(torch.load(MEILLEUR, map_location=app)); modele.eval().cpu()
torch.onnx.export(modele, torch.zeros(1, 3, TAILLE, TAILLE), f'{DOSSIER}/promopro-plan3d.onnx', input_names=['image'], output_names=['logits'], opset_version=17, dynamic_axes={'image': {0: 'lot'}, 'logits': {0: 'lot'}})
print('ONNX écrit :', f'{DOSSIER}/promopro-plan3d.onnx', os.path.getsize(f'{DOSSIER}/promopro-plan3d.onnx') // 1_000_000, 'Mo')

# Test : déposez villa_plan_2d.png, puis réutilisez les cellules 5 et 6 du carnet 01 (inférence et JSON) avec ce modèle
import onnxruntime as ort
from google.colab import files
chemin = next(iter(files.upload()))
img = cv2.cvtColor(cv2.imread(chemin), cv2.COLOR_BGR2RGB); H, W = img.shape[:2]
x = cv2.resize(img, (TAILLE, TAILLE)).astype(np.float32) / 255; x = ((x - moy.numpy().transpose(1, 2, 0)) / ecart.numpy().transpose(1, 2, 0)).transpose(2, 0, 1)[None].astype(np.float32)
sess = ort.InferenceSession(f'{DOSSIER}/promopro-plan3d.onnx'); logits = sess.run(None, {'image': x})[0]
masque = cv2.resize(logits[0].argmax(0).astype(np.uint8), (W, H), interpolation=cv2.INTER_NEAREST)
print('pixels par classe :', {c: int((masque == c).sum()) for c in range(4)})
couleurs = np.array([[255, 255, 255], [20, 20, 20], [230, 120, 0], [0, 120, 230]], dtype=np.uint8)
cv2.imwrite('masque_promopro.png', cv2.cvtColor((0.55 * img + 0.45 * couleurs[masque]).astype(np.uint8), cv2.COLOR_RGB2BGR))
from IPython.display import Image, display; display(Image('masque_promopro.png', width=700)); files.download('masque_promopro.png')